# Hutch++ Code:

## **Algorithm 1 Hutch++**

-------------------------

**Input:** Matrix–vector multiplication oracle for matrix $(A \in \mathbb{R}^{d \times d})$.  Number of queries $m$.

------------------------

**Output:** Approximation to $\mathrm{tr}(A)$.

1. Sample $S \in \mathbb{R}^{d \times \frac{m}{3}}$ and $G \in \mathbb{R}^{d \times \frac{m}{3}}$ with i.i.d. $\{+1, -1\}$ entries.  
2. Compute an orthonormal basis $Q \in \mathbb{R}^{d \times \frac{m}{3}}$ for the span of $AS$ (e.g., via QR decomposition).  
3. **Return**  
   $$
   \text{Hutch}^{++}(A) = 
   \operatorname{tr}(Q^T A Q)
   + \frac{3}{m}\operatorname{tr}\!\left(G^T (I - QQ^T) A (I - QQ^T) G \right).
   $$
   
--------------------------


In [1]:
import numpy as np
import scipy.linalg as la
from scipy.sparse.linalg import LinearOperator
import torch
np.random.seed(0)     

In [2]:
d = 100
m = 1000

In [3]:
M = np.random.randn(d, d)
A_mat = M.T @ M   # PSD
# But here A_mat does not have to be PSD, it can be any matrix.
print("The true trace is:", np.trace(A_mat))
def A_mv(v):
    return A_mat @ v
A = LinearOperator((d, d), matvec=A_mv)
def Hutch_pplus(A, m, d):
    S = np.random.choice([-1, 1], size = (d, m//3))
    G = np.random.choice([-1, 1], size = (d, m//3))
    # The reason we use column_stack is because we want to compute A@s for each s in S, 
    # and we want to stack the results as columns in a matrix. 
    # This way, we can easily compute the trace of Q.T@AQ later on.
    AS = np.column_stack([A@s for s in S.T]) 
    Q, _ = np.linalg.qr(AS)
    AQ = np.column_stack([A@q for q in Q.T])
    I = np.eye(d)
    B = I - Q@Q.T
    BG = B@G
    ABG = np.column_stack([A@bg for bg in BG.T])
    return  (np.trace(Q.T@AQ) + (3/m) * np.trace(G.T@B@ABG))
#The reason we can multiply without knowing v is because we are using a LinearOperator, 
# which allows us to define the matrix-vector product without explicitly forming the matrix. 
# This is useful for large matrices where forming the full matrix would be computationally 
# expensive or memory-intensive.
if __name__ == "__main__":
    print(Hutch_pplus(A, m, d))




The true trace is: 9756.077773866702
9756.077773866704


Find some large dataset and try to use the Hutch++ to implement the trace of the large data matrix.

## **Algorithm 2 — NA-Hutch++ (Non-Adaptive variant of Hutch++)**

**Input:**  
Matrix–vector multiplication oracle for a Positive Semidefinite matrix $ A \in \mathbb{R}^{d \times d} $.  
Number of queries $ m $.

**Output:**  
Approximation to $ \mathrm{tr}(A) $.

---

1. Fix constants $ c_1, c_2, c_3 $ such that  
   $$
   c_1 < c_2,\qquad c_1 + c_2 + c_3 = 1.
   $$

2. Sample  
   $$
   S \in \mathbb{R}^{d \times c_1 m}, \qquad
   R \in \mathbb{R}^{d \times c_2 m}, \qquad
   G \in \mathbb{R}^{d \times c_3 m}
   $$
   with i.i.d. $ \{+1, -1\} $ entries.

3. Compute  
   $$
   Z = A R, \qquad
   W = A S.
   $$

4. Return  
   $$
   \text{NA-Hutch}^{++}(A)
   = \mathrm{tr}\!\left((S^\top Z)^{+}(W^\top Z)\right)
   + \frac{1}{c_3 m}
     \left[
       \mathrm{tr}(G^\top A G)
       - \mathrm{tr}\!\left(G^\top Z (S^\top Z)^{+} W^\top G\right)
     \right].
   $$
---------------
   Purpose: To eliminate adaptivity and allow all matrix-vector products with A to be computed in one batch, before doing any QR or pseudoinverses.


In [4]:
M = np.random.randn(d, d)
A_mat = M.T @ M   # PSD
# Here, A must be PSD for NA-Hutch++ to work.
print("The true trace is:", np.trace(A_mat))
def A_mv(v):
    return A_mat @ v
A = LinearOperator((d, d), matvec=A_mv)
def NA_Hutch_pplus(A, m, d):
    c1 = 0.3; c2 = 0.3
    c3 = 1 - c1 - c2
    m1 = int(m * c1)
    m2 = int(m * c2)
    m3 = m - m1 - m2 
    S = np.random.choice([-1, 1], size = (d, m1))
    R = np.random.choice([-1, 1], size = (d, m2))
    G = np.random.choice([-1, 1], size = (d, m3))
    # The reason we use column_stack is because we want to compute A@s for each s in S, 
    # and we want to stack the results as columns in a matrix. 
    # This way, we can easily compute the trace of Q.T@AQ later on.
    AS = np.column_stack([A@s for s in S.T]) 
    W = AS
    AR = np.column_stack([A@r for r in R.T])
    Z = AR
    AG = np.column_stack([A@g for g in G.T])
    I = np.eye(d)
    STZ_pinv = la.pinv(S.T@Z)
    STZ_pinv_WT_G = np.column_stack([STZ_pinv@W.T@g for g in G.T])
    STZ_pinv_WT_Z = np.column_stack([STZ_pinv@W.T@z for z in Z.T])
    return  (np.trace(STZ_pinv_WT_Z) + 1/(c3*m)*(np.trace(G.T@AG) - np.trace(G.T@Z@STZ_pinv_WT_G)))
#The reason we can multiply without knowing v is because we are using a LinearOperator, 
# which allows us to define the matrix-vector product without explicitly forming the matrix. 
# This is useful for large matrices where forming the full matrix would be computationally 
# expensive or memory-intensive.
if __name__ == "__main__":
    print(NA_Hutch_pplus(A, m, d))




The true trace is: 9947.917640864547
9947.917640864516


## **Algorithm 3 — Gaussian-Hutch++ (Gaussian Variant of Hutch++)**

**Input:**  
Matrix–vector multiplication oracle for matrix $ A \in \mathbb{R}^{d \times d} $.  
Number of queries $ m $.

**Output:**  
Approximation to $ \mathrm{tr}(A) $.

---

1. Sample  
   $$
   S \in \mathbb{R}^{d \times \frac{m+2}{4}}
   $$
   with i.i.d. $ \mathcal{N}(0,1) $ entries, and  
   $$
   G \in \mathbb{R}^{d \times \frac{m-2}{2}}
   $$
   with i.i.d. $\{+1,-1\}$ entries.

2. Compute an orthonormal basis  
   $$
   Q \in \mathbb{R}^{d \times \frac{m+2}{4}}
   $$
   for the span of $ AS $ (e.g., via QR decomposition).

3. Return  
   $$
   \text{Gaussian-Hutch}^{++}(A)
   = \operatorname{tr}(Q^{T} A Q)
   + \frac{2}{m - 2}\,
     \operatorname{tr}\!\left(G^{T}(I - QQ^{T}) A (I - QQ^{T}) G\right).
   $$
-----------#
Purpose: Provide a Gaussian version of Hutch++ that allows rigorous variance bounds with explicit constants and optimally chosen sketch sizes.

In [5]:
M = np.random.randn(d, d)
A_mat = M.T @ M   # PSD
print("The true trace is:", np.trace(A_mat))
def A_mv(v):
    return A_mat @ v
A = LinearOperator((d, d), matvec=A_mv)
def Gaussian_Hutch_pplus(A, m, d):
    S = np.random.normal(loc=0, scale=1, size = (d, (m+2)//4))

    G = np.random.choice([-1, 1], size = (d, (m-2)//2))
    # The reason we use column_stack is because we want to compute A@s for each s in S, 
    # and we want to stack the results as columns in a matrix. 
    # This way, we can easily compute the trace of Q.T@AQ later on.
    AS = np.column_stack([A@s for s in S.T]) 
    Q, _ = np.linalg.qr(AS)
    AQ = np.column_stack([A@q for q in Q.T])
    I = np.eye(d)
    B = I - Q@Q.T
    BG = B@G
    ABG = np.column_stack([A@bg for bg in BG.T])
    return  (np.trace(Q.T@AQ) + 2/(m-2) * np.trace(G.T@B@ABG))
#The reason we can multiply without knowing v is because we are using a LinearOperator, 
# which allows us to define the matrix-vector product without explicitly forming the matrix. 
# This is useful for large matrices where forming the full matrix would be computationally 
# expensive or memory-intensive.
if __name__ == "__main__":
    print(Gaussian_Hutch_pplus(A, m, d))




The true trace is: 10051.069415677557
10051.069415677555
